In [1]:
import numpy as np 
import pandas as pd 

import warnings 
warnings.filterwarnings('ignore')

In [2]:
import tensorflow as tf
from tensorflow import keras 
from keras.models import Sequential
from keras.layers import Dense, BatchNormalization, Dropout, Activation, Input
from keras.regularizers import l1_l2
from keras.optimizers import Adam, RMSprop, SGD
from keras.callbacks import EarlyStopping

import optuna

In [3]:
X_train_t = pd.read_csv(r"C:\Users\laksh\DL\Split data\X_train_t.csv")
X_val_t = pd.read_csv(r"C:\Users\laksh\DL\Split data\X_val_t.csv")
X_test_t = pd.read_csv(r"C:\Users\laksh\DL\Split data\X_test_t.csv")

y_train = pd.read_csv(r"C:\Users\laksh\DL\Split data\y_train.csv")
y_val = pd.read_csv(r"C:\Users\laksh\DL\Split data\y_val.csv")
y_test = pd.read_csv(r"C:\Users\laksh\DL\Split data\y_test.csv")

In [4]:
# X_test_t.drop(columns=['ohe__reserved_room_type_P', 'ohe__assigned_room_type_P'],inplace=True)

In [5]:
X_train_t.shape, X_val_t.shape, X_test_t.shape

((55641, 66), (13911, 66), (17388, 66))

In [6]:
y_train.shape, y_val.shape, y_test.shape

((55641, 1), (13911, 1), (17388, 1))

In [7]:
model = Sequential()

In [8]:
model.add(Dense(units=64, input_shape=(X_train_t.shape[1],)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))

# Hiddden layer 2
model.add(Dense(units=32)) 
model.add(BatchNormalization()) 
model.add(Activation('relu')) 
# model.add(Dropout(0.2))

# Hidden layer 3
model.add(Dense(units=32)) 
model.add(BatchNormalization()) 
model.add(Activation('relu')) 
# model.add(Dropout(0.2))

# Output layer 
model.add(Dense(units=1, activation='sigmoid'))

In [9]:
model.compile(loss='binary_crossentropy', metrics=['accuracy'], optimizer='Adam')

In [10]:
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [11]:
X_train_t.shape, y_train.shape

((55641, 66), (55641, 1))

In [12]:
model.fit(X_train_t,y_train,
         batch_size=128,
         epochs=50,
         validation_data=(X_val_t,y_val),
         callbacks=[es],
         verbose=1)

Epoch 1/50
435/435 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7026 - loss: 0.5767 - val_accuracy: 0.7928 - val_loss: 0.4591
Epoch 2/50
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7876 - loss: 0.4503 - val_accuracy: 0.8017 - val_loss: 0.4287
Epoch 3/50
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7964 - loss: 0.4297 - val_accuracy: 0.8038 - val_loss: 0.4206
Epoch 4/50
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7992 - loss: 0.4191 - val_accuracy: 0.8104 - val_loss: 0.4131
Epoch 5/50
435/435 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8040 - loss: 0.4130 - val_accuracy: 0.8118 - val_loss: 0.4087
Epoch 6/50
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8066 - loss: 0.4082 - val_accuracy: 0.8135 - val_loss: 0.4051
Epoch 7/50
435/435 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8102 - loss: 0.4029 - val_accuracy: 0.8142 - val_loss: 0.4024
Epoch 8/50
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8080 - loss: 0.4023 - val_accuracy: 0.

In [13]:
model.evaluate(X_test_t,y_test) 

544/544 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8215 - loss: 0.3817


[0.381680965423584, 0.8215435743331909]

# Hyper Parameter Tuning

In [14]:
def objective(trial):
    lr_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1, 4)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'RMSprop', 'SGD'])
    activation = trial.suggest_categorical('activation', ['tanh', 'relu'])
    batch_size1 = trial.suggest_categorical('batch_size', [32, 64, 128, 256, 512])

    model = Sequential()
    model.add(Input(shape=(X_train_t.shape[1],)))
    for i in range(n_layers):
        units = trial.suggest_int(f'units{i}', 8, 96)
        dropout = trial.suggest_float(f'dropout{i}', 0.0, 0.5)
        reg = trial.suggest_float(f'reg{i}', 1e-5, 1e-2, log=True)
        model.add(Dense(units, activation=activation, kernel_regularizer=l1_l2(l1=reg, l2=reg)))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))
    model.add(Dense(1, activation='sigmoid'))

    opt_map = {
        'Adam': Adam(learning_rate=lr_rate),
        'RMSprop': RMSprop(learning_rate=lr_rate),
        'SGD': SGD(learning_rate=lr_rate)
    }
    model.compile(optimizer=opt_map[optimizer_name], loss='binary_crossentropy',
                  metrics=[tf.keras.metrics.Recall(name='recall')])

    es = EarlyStopping(monitor='val_recall', mode='max', patience=5, restore_best_weights=True)
    history = model.fit(
        X_train_t, y_train,
        epochs=30, batch_size=batch_size1,
        validation_data=(X_val_t, y_val),   # validation set, NOT test
        callbacks=[es], verbose=0
    )
    return max(history.history['val_recall'])

In [15]:
def objective(trial):

    lr_rate = trial.suggest_float(
        'learning_rate',
        1e-4,
        5e-3,
        log=True
    )

    n_layers = trial.suggest_int(
        'n_layers',
        2,
        4
    )

    optimizer_name = trial.suggest_categorical(
        'optimizer',
        ['Adam', 'RMSprop']
    )

    activation = trial.suggest_categorical(
        'activation',
        ['relu', 'tanh']
    )

    batch_size1 = trial.suggest_categorical(
        'batch_size',
        [64, 128, 256]
    )

    model = Sequential()

    model.add(Input(shape=(X_train_t.shape[1],)))

    for i in range(n_layers):

        units = trial.suggest_categorical(
            f'units{i}',
            [64, 128, 256]
        )

        dropout = trial.suggest_float(
            f'dropout{i}',
            0.0,
            0.3
        )

        reg = trial.suggest_float(
            f'reg{i}',
            1e-6,
            1e-4,
            log=True
        )

        model.add(
            Dense(
                units,
                activation=activation,
                kernel_regularizer=l1_l2(
                    l1=reg,
                    l2=reg
                )
            )
        )

        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    model.add(Dense(1, activation='sigmoid'))

    opt_map = {
        'Adam': Adam(learning_rate=lr_rate),
        'RMSprop': RMSprop(learning_rate=lr_rate)
    }

    model.compile(
        optimizer=opt_map[optimizer_name],
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

    es = EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=8,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_t,
        y_train,
        validation_data=(X_val_t, y_val),
        epochs=50,
        batch_size=batch_size1,
        callbacks=[es],
        verbose=0
    )

    return max(history.history['val_accuracy'])

In [16]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=21))
study.optimize(objective, n_trials=5, show_progress_bar=True)  

[I 2026-07-03 19:24:26,336] A new study created in memory with name: no-name-c8f447fc-a7e9-42d5-96e2-87552d6c7d7c


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-07-03 19:27:29,887] Trial 0 finished with value: 0.8190640211105347 and parameters: {'learning_rate': 0.00012099909190486929, 'n_layers': 2, 'optimizer': 'Adam', 'activation': 'relu', 'batch_size': 128, 'units0': 256, 'dropout0': 0.03997215577552432, 'reg0': 2.2711683274883174e-06, 'units1': 128, 'dropout1': 0.29114553760474726, 'reg1': 3.3006927980408684e-05}. Best is trial 0 with value: 0.8190640211105347.
[I 2026-07-03 19:31:21,386] Trial 1 finished with value: 0.8188483715057373 and parameters: {'learning_rate': 0.00044960308102120935, 'n_layers': 3, 'optimizer': 'Adam', 'activation': 'tanh', 'batch_size': 64, 'units0': 256, 'dropout0': 0.1380421001378928, 'reg0': 1.226695349735026e-05, 'units1': 128, 'dropout1': 0.25478781401736506, 'reg1': 3.428022068943908e-06, 'units2': 128, 'dropout2': 0.12310625357184563, 'reg2': 4.171719495598641e-05}. Best is trial 0 with value: 0.8190640211105347.
[I 2026-07-03 19:37:02,164] Trial 2 finished with value: 0.819998562335968 and parame

In [17]:
study.best_params

{'learning_rate': 0.002129133206108677,
 'n_layers': 2,
 'optimizer': 'Adam',
 'activation': 'tanh',
 'batch_size': 256,
 'units0': 64,
 'dropout0': 0.04525529640481044,
 'reg0': 8.384254227036188e-06,
 'units1': 64,
 'dropout1': 0.13883967834830424,
 'reg1': 6.816277159257633e-05}

In [18]:
# Input Layer + Hidden Layer 1
model.add(Dense(input_shape=(X_train_t.shape[1],), units=256 ,kernel_initializer='he_normal',
                kernel_regularizer=l1_l2(l1=9.410229566634376e-05,l2 = 9.410229566634376e-05)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2955656520341639))

# Hidden Layer 2
model.add(Dense(units=64 , kernel_initializer='he_normal',kernel_regularizer=l1_l2(l1=1.2863587109512727e-05,l2 = 1.2863587109512727e-05)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.1758430856697497))

# Hidden Layer 3
model.add(Dense(units=64 , kernel_initializer='he_normal',kernel_regularizer=l1_l2(l1=1.2665491472862916e-06,l2 = 1.2665491472862916e-06)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.28824805617934074))

# Hidden Layer 4
model.add(Dense(units=256 , kernel_initializer='he_normal',kernel_regularizer=l1_l2(l1=8.682467669301482e-05,l2 = 8.682467669301482e-05)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2895983419205378))

# Output Layer 
model.add(Dense(units=1, activation='sigmoid'))

In [19]:
model.compile(metrics=['accuracy'], loss='binary_crossentropy', optimizer=Adam(learning_rate=0.00034528176998195744))

In [20]:
es = EarlyStopping(monitor ='val_loss', patience=5, restore_best_weights=True)

In [21]:
model.fit(X_train_t, y_train,
         batch_size=256,
         epochs=40,
         validation_data=(X_val_t,y_val), 
         callbacks=[es],
         verbose=1)

Epoch 1/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.8064 - loss: 0.7502 - val_accuracy: 0.8197 - val_loss: 0.7148
Epoch 2/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.8187 - loss: 0.6856 - val_accuracy: 0.8206 - val_loss: 0.6732
Epoch 3/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8226 - loss: 0.6492 - val_accuracy: 0.8214 - val_loss: 0.6516
Epoch 4/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8257 - loss: 0.6207 - val_accuracy: 0.8203 - val_loss: 0.6286
Epoch 5/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8264 - loss: 0.5929 - val_accuracy: 0.8213 - val_loss: 0.6050
Epoch 6/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8268 - loss: 0.5707 - val_accuracy: 0.8220 - val_loss: 0.5891
Epoch 7/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8279 - loss: 0.5517 - val_accuracy: 0.8219 - val_loss: 0.5718
Epoch 8/40
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8291 - loss: 0.5313 - val_acc

In [22]:
model.evaluate(X_test_t,y_test)

544/544 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8222 - loss: 0.3927


[0.39270469546318054, 0.8221762180328369]

In [23]:
model.save('model.keras')

In [24]:
import sklearn
print(sklearn.__version__)

1.8.0
